# Embedded C — Part 3: Memory, Fixed-Point Math and Reliability

A microcontroller might have 32 KB of Flash and 4 KB of RAM, no floating-point hardware, and a requirement to run for years without a reboot. This notebook covers the techniques that make that possible:

1. Where your variables actually live (Flash vs RAM sections)
2. A fixed-block **memory pool** instead of `malloc`
3. **Fixed-point** arithmetic instead of `float`
4. **CRC** checksums to detect corrupted data
5. Compile-time and run-time **checks** (`_Static_assert`, `assert`)
6. Optimisation levels and code size

## Files created

```
/content/embedded_c/part3/
├── include/
│   ├── mem_pool.h       fixed-block allocator interface
│   └── fixed.h          Q16.16 fixed-point type and helpers
└── src/
    ├── sections.c       .text / .rodata / .data / .bss in practice
    ├── mem_pool.c       O(1), fragmentation-free allocator
    ├── pool_demo.c      using the pool for messages
    ├── fixed_demo.c     ADC -> temperature without float
    ├── crc.c            CRC-8 and CRC-16, bitwise and table-driven
    └── checks.c         _Static_assert and assert
```

## 0. Setup

In [ ]:
!mkdir -p /content/embedded_c/part3/include /content/embedded_c/part3/src
!gcc --version | head -n 1

---
## 1. Where variables live: memory sections

When firmware is linked, every variable and function goes into a **section**, and the linker script maps each section to Flash or RAM:

| Section | Contains | Stored in |
|---|---|---|
| `.text` | machine code | Flash |
| `.rodata` | `const` data: lookup tables, strings | Flash |
| `.data` | globals with a non-zero initial value | Flash (initial values) **and** RAM (copied at startup) |
| `.bss` | globals that start at zero | RAM only, zeroed at startup |
| stack | local variables, return addresses | RAM, grows at runtime |
| heap | `malloc` memory (if used at all) | RAM |

**Practical consequences:**

- Mark big tables `const` so they stay in Flash and cost no RAM.
- A large local array (`uint8_t buf[2048];` inside a function) can silently **overflow the stack**. Make it `static` or global so its size is fixed and visible at link time.
- `size` and `nm` show exactly how much each section uses, the same numbers your IDE reports after building firmware.

In [ ]:
%%writefile /content/embedded_c/part3/src/sections.c
#include <stdio.h>
#include <stdint.h>

const uint8_t  sine_table[8] = {0, 49, 90, 117, 127, 117, 90, 49};  /* .rodata -> Flash only */
uint32_t       boot_count = 5;          /* .data -> Flash copy + RAM           */
uint32_t       sample_buffer[256];      /* .bss  -> 1 KB of RAM, zeroed        */
static uint8_t private_state;           /* .bss, visible only in this file     */

int main(void)
{
    uint32_t local = 3;                 /* stack */
    private_state++;
    sample_buffer[0] = sine_table[4] + boot_count + local + private_state;
    printf("sample_buffer[0] = %u\n", (unsigned)sample_buffer[0]);
    return 0;
}

In [ ]:
!cd /content/embedded_c/part3 && gcc -std=c11 -O2 -c src/sections.c -o sections.o && echo "=== section sizes (bytes) ===" && size sections.o && echo && echo "=== symbols: size, section letter, name ===" && nm -S --size-sort sections.o

`size` counts `.rodata` in the `text` column, since both end up in Flash. In `nm`, the letter shows the section: **T** = text, **R** = read-only data, **D** = data, **B**/**b** = bss (lowercase means `static`, i.e. private). `sample_buffer` takes `0x400` = 1024 bytes of `.bss`: on a chip with 4 KB of RAM, that's a quarter of everything.

---
## 2. A memory pool instead of `malloc`

`malloc` is often banned in embedded code (MISRA C forbids it) because:

- its timing is unpredictable, which is a problem for real-time code;
- after many allocate/free cycles, RAM becomes **fragmented**: there's enough free memory in total, but no single gap is big enough;
- failure shows up at runtime, possibly months after deployment.

A **fixed-block pool** avoids all three. Memory is a static array of equal-sized blocks set aside at compile time. Free blocks are chained into a **free list**, and the link pointer is stored *inside* the free blocks themselves, so the bookkeeping costs no extra RAM. Allocating and freeing each take a handful of instructions, always the same number.

In [ ]:
%%writefile /content/embedded_c/part3/include/mem_pool.h
#ifndef MEM_POOL_H
#define MEM_POOL_H

#include <stddef.h>
#include <stdint.h>

#define POOL_BLOCK_SIZE  32u     /* bytes per block  */
#define POOL_BLOCK_COUNT 8u      /* number of blocks */

void   pool_init(void);
void  *pool_alloc(void);          /* O(1); returns NULL when the pool is empty */
void   pool_free(void *block);    /* O(1) */
size_t pool_free_count(void);

#endif /* MEM_POOL_H */

In [ ]:
%%writefile /content/embedded_c/part3/src/mem_pool.c
#include "mem_pool.h"

typedef union block {
    union block *next;                    /* used while the block is free   */
    uint8_t      bytes[POOL_BLOCK_SIZE];  /* used while the block is in use */
    max_align_t  align;                   /* forces alignment safe for any type */
} block_t;

static block_t  pool[POOL_BLOCK_COUNT];   /* fixed at compile time: shows up in .bss */
static block_t *free_list;
static size_t   free_count;

void pool_init(void)
{
    for (size_t i = 0; i < POOL_BLOCK_COUNT - 1; i++)
        pool[i].next = &pool[i + 1];      /* chain every block to the next */
    pool[POOL_BLOCK_COUNT - 1].next = NULL;
    free_list  = &pool[0];
    free_count = POOL_BLOCK_COUNT;
}

void *pool_alloc(void)
{
    if (free_list == NULL) return NULL;   /* empty: fail fast and predictably */
    block_t *b = free_list;
    free_list  = b->next;                 /* unlink the first free block */
    free_count--;
    return b;
}

void pool_free(void *ptr)
{
    if (ptr == NULL) return;
    block_t *b = ptr;
    if (b < &pool[0] || b > &pool[POOL_BLOCK_COUNT - 1]) return;   /* not one of ours */
    b->next   = free_list;                /* push it back on the front */
    free_list = b;
    free_count++;
}

size_t pool_free_count(void)
{
    return free_count;
}

In [ ]:
%%writefile /content/embedded_c/part3/src/pool_demo.c
#include <stdio.h>
#include <string.h>
#include "mem_pool.h"

typedef struct {
    uint8_t  id;
    uint16_t length;
    char     payload[24];
} message_t;

/* Fails the BUILD if someone later makes message_t too big for a block */
_Static_assert(sizeof(message_t) <= POOL_BLOCK_SIZE, "message_t must fit in one pool block");

int main(void)
{
    pool_init();
    printf("free blocks at start: %zu\n", pool_free_count());

    message_t *msgs[10] = {0};
    int got = 0;
    for (int i = 0; i < 10; i++) {
        msgs[i] = pool_alloc();
        if (msgs[i] == NULL) {
            printf("alloc #%d failed: pool empty, handled cleanly\n", i);
            continue;
        }
        msgs[i]->id = (uint8_t)i;
        snprintf(msgs[i]->payload, sizeof msgs[i]->payload, "sensor reading %d", i * 11);
        msgs[i]->length = (uint16_t)strlen(msgs[i]->payload);
        got++;
    }
    printf("allocated %d messages, free blocks now: %zu\n", got, pool_free_count());
    printf("msg 3: id=%u len=%u \"%s\"\n", msgs[3]->id, msgs[3]->length, msgs[3]->payload);

    pool_free(msgs[3]);
    pool_free(msgs[5]);
    printf("after freeing 2 messages: free blocks = %zu\n", pool_free_count());

    message_t *again = pool_alloc();
    printf("next alloc reuses %s\n",
           (void *)again == (void *)msgs[5] ? "msg 5's block (last freed, first reused)" : "another block");
    return 0;
}

In [ ]:
!cd /content/embedded_c/part3 && gcc -Wall -Wextra -std=c11 -Iinclude src/mem_pool.c src/pool_demo.c -o pool_demo && ./pool_demo

Real systems often keep several pools (say 16-, 64- and 256-byte blocks) and pick the smallest that fits. RTOSes like FreeRTOS and Zephyr provide exactly this kind of allocator.

---
## 3. Fixed-point arithmetic

Many microcontrollers (Cortex-M0/M3, AVR, most 8-bit parts) have **no floating-point unit**. Every `float` operation then becomes a slow software routine, and the math library can add kilobytes to the firmware.

**Fixed-point** stores fractions in ordinary integers with an agreed scale. In **Q16.16**, a 32-bit integer holds 16 integer bits and 16 fraction bits, so the real value is `raw / 65536`:

| Real value | Raw `int32_t` |
|---|---|
| 1.0 | 65536 |
| 2.5 | 163840 |
| −0.25 | −16384 |

- **Add / subtract:** just use `+` and `-`.
- **Multiply:** `(a * b) >> 16`, computed in 64 bits so the intermediate product doesn't overflow.
- **Divide:** `(a << 16) / b`, again in 64 bits.

The example converts raw 12-bit ADC readings from a temperature sensor (10 mV per °C, 3.3 V reference) using only integer math, and compares the results to `double`.

**Precision tip:** combine constants **before** converting them to fixed-point. `330/4095` as one Q16 constant keeps far more accuracy than converting `3.3/4095` (a tiny number) and then multiplying by 100.

In [ ]:
%%writefile /content/embedded_c/part3/include/fixed.h
#ifndef FIXED_H
#define FIXED_H

#include <stdint.h>

/* Q16.16 fixed point: 16 integer bits . 16 fractional bits.  1.0 == 65536 */
typedef int32_t q16_t;

#define Q16_SHIFT        16
#define Q16_ONE          ((q16_t)1 << Q16_SHIFT)
#define Q16_FROM_INT(x)  ((q16_t)(x) * Q16_ONE)

/* Rounded conversion. Use on constants only: the compiler evaluates it,
   so no floating-point code ends up in the firmware. */
#define Q16_FROM_FLOAT(f) ((q16_t)((f) * Q16_ONE + ((f) >= 0 ? 0.5 : -0.5)))

/* For printing and debugging on the PC only */
#define Q16_TO_DOUBLE(q) ((double)(q) / Q16_ONE)

static inline q16_t q16_mul(q16_t a, q16_t b)
{
    return (q16_t)(((int64_t)a * b) >> Q16_SHIFT);    /* widen to 64 bits first */
}

static inline q16_t q16_div(q16_t a, q16_t b)
{
    return (q16_t)(((int64_t)a << Q16_SHIFT) / b);
}

static inline int32_t q16_to_int_round(q16_t q)
{
    return (q + (Q16_ONE / 2)) >> Q16_SHIFT;
}

#endif /* FIXED_H */

In [ ]:
%%writefile /content/embedded_c/part3/src/fixed_demo.c
#include <stdio.h>
#include "fixed.h"

int main(void)
{
    /* degrees C per ADC count: (3.3 V / 4095 counts) * (100 C per V) = 330 / 4095 */
    const q16_t DEG_PER_COUNT = Q16_FROM_FLOAT(330.0 / 4095.0);
    const q16_t NINE_FIFTHS   = Q16_FROM_FLOAT(1.8);
    const q16_t THIRTY_TWO    = Q16_FROM_INT(32);

    const uint16_t readings[] = { 0, 310, 775, 1241, 2048, 4095 };

    printf("  ADC | fixed-point C | double C | error    | F (fixed) | rounded C\n");
    printf("------+---------------+----------+----------+-----------+----------\n");
    for (size_t i = 0; i < sizeof readings / sizeof readings[0]; i++) {
        q16_t  degC = (q16_t)readings[i] * DEG_PER_COUNT;        /* integer * Q16 = Q16 */
        q16_t  degF = q16_mul(degC, NINE_FIFTHS) + THIRTY_TWO;
        double ref  = readings[i] * 3.3 / 4095.0 * 100.0;

        printf(" %4u | %13.4f | %8.4f | %+8.4f | %9.3f | %5d\n",
               readings[i], Q16_TO_DOUBLE(degC), ref, Q16_TO_DOUBLE(degC) - ref,
               Q16_TO_DOUBLE(degF), (int)q16_to_int_round(degC));
    }

    q16_t a = Q16_FROM_FLOAT(7.5), b = Q16_FROM_FLOAT(2.25);
    printf("\n7.5 + 2.25 = %.4f\n", Q16_TO_DOUBLE(a + b));
    printf("7.5 * 2.25 = %.4f\n", Q16_TO_DOUBLE(q16_mul(a, b)));
    printf("7.5 / 2.25 = %.4f\n", Q16_TO_DOUBLE(q16_div(a, b)));
    printf("7.5 is stored as the plain integer %d (0x%08X)\n", (int)a, (unsigned)a);
    return 0;
}

In [ ]:
!cd /content/embedded_c/part3 && gcc -Wall -Wextra -std=c11 -Iinclude src/fixed_demo.c -o fixed_demo && ./fixed_demo

The error stays under two hundredths of a degree across the full range, far below what a real sensor can resolve, and every operation is an integer instruction. (`printf` with `%f` here is just for displaying results on the PC. Firmware would print the integer and fraction parts separately.)

Other common formats are **Q15** (`int16_t`, range −1 to almost +1, used by DSP and audio code; ARM's CMSIS-DSP library is built on it) and **Q8.8** for 8/16-bit MCUs.

---
## 4. CRC: detecting corrupted data

Electrical noise flips bits on UART, SPI, I²C and radio links, and Flash memory can wear out. A **CRC** (Cyclic Redundancy Check) is a short checksum the sender appends to the data. The receiver recomputes it, and if the values differ, the data is corrupted. It's used in Modbus, CAN, USB, Ethernet, SD cards, and firmware-update images.

There are two ways to implement the same CRC:

- **Bit by bit:** tiny code, 8 loop steps per byte.
- **Table-driven:** a 256-entry lookup table (512 bytes for CRC-16, best placed in Flash as `const`) and one step per byte. It's a classic **speed vs. memory** trade-off.

Every CRC variant has a published **check value**: the CRC of the ASCII string `"123456789"`. Always test your implementation against it.

In [ ]:
%%writefile /content/embedded_c/part3/src/crc.c
#include <stdio.h>
#include <stdint.h>
#include <string.h>

/* CRC-8 (polynomial 0x07, init 0x00): bit by bit, small and slow */
uint8_t crc8(const uint8_t *data, size_t len)
{
    uint8_t crc = 0x00;
    for (size_t i = 0; i < len; i++) {
        crc ^= data[i];
        for (int b = 0; b < 8; b++)
            crc = (crc & 0x80) ? (uint8_t)((crc << 1) ^ 0x07) : (uint8_t)(crc << 1);
    }
    return crc;
}

/* CRC-16/CCITT-FALSE (polynomial 0x1021, init 0xFFFF): bit by bit */
uint16_t crc16_bitwise(const uint8_t *data, size_t len)
{
    uint16_t crc = 0xFFFF;
    for (size_t i = 0; i < len; i++) {
        crc ^= (uint16_t)(data[i] << 8);
        for (int b = 0; b < 8; b++)
            crc = (crc & 0x8000) ? (uint16_t)((crc << 1) ^ 0x1021) : (uint16_t)(crc << 1);
    }
    return crc;
}

/* Same CRC with a lookup table: 512 bytes of memory, 8x fewer steps.
   In firmware you'd generate this table once and paste it in as a const array. */
static uint16_t crc16_table[256];

void crc16_build_table(void)
{
    for (int i = 0; i < 256; i++) {
        uint16_t crc = (uint16_t)(i << 8);
        for (int b = 0; b < 8; b++)
            crc = (crc & 0x8000) ? (uint16_t)((crc << 1) ^ 0x1021) : (uint16_t)(crc << 1);
        crc16_table[i] = crc;
    }
}

uint16_t crc16_table_driven(const uint8_t *data, size_t len)
{
    uint16_t crc = 0xFFFF;
    for (size_t i = 0; i < len; i++)
        crc = (uint16_t)((crc << 8) ^ crc16_table[((crc >> 8) ^ data[i]) & 0xFF]);
    return crc;
}

int main(void)
{
    const uint8_t *check = (const uint8_t *)"123456789";   /* the standard test string */
    size_t n = 9;
    crc16_build_table();

    printf("CRC-8          = 0x%02X   (expected 0xF4)\n",   crc8(check, n));
    printf("CRC-16 bitwise = 0x%04X (expected 0x29B1)\n", crc16_bitwise(check, n));
    printf("CRC-16 table   = 0x%04X (expected 0x29B1)\n", crc16_table_driven(check, n));

    /* Typical use: sender appends the CRC to a packet (big-endian) */
    uint8_t packet[6] = { 0x01, 0x03, 0x00, 0x2A };
    uint16_t crc = crc16_table_driven(packet, 4);
    packet[4] = (uint8_t)(crc >> 8);
    packet[5] = (uint8_t)(crc & 0xFF);
    printf("\nsent packet    : ");
    for (int i = 0; i < 6; i++) printf("%02X ", packet[i]);

    /* Receiver side, before and after a 1-bit error on the wire */
    for (int corrupt = 0; corrupt <= 1; corrupt++) {
        if (corrupt) packet[2] ^= 0x10;           /* flip one bit */
        uint16_t received = (uint16_t)((packet[4] << 8) | packet[5]);
        uint16_t computed = crc16_table_driven(packet, 4);
        printf("\n%-15s: computed 0x%04X, received 0x%04X -> %s",
               corrupt ? "after bit flip" : "clean packet", computed, received,
               computed == received ? "OK, accept" : "CORRUPTED, discard");
    }
    printf("\n");
    return 0;
}

In [ ]:
!cd /content/embedded_c/part3 && gcc -Wall -Wextra -std=c11 src/crc.c -o crc && ./crc

---
## 5. Catching bugs early: `_Static_assert` and `assert`

The cheapest bug is one that stops the build.

- `_Static_assert(condition, "message")` is checked by the **compiler**. It costs nothing at runtime. Use it to guarantee struct sizes that must match a protocol, buffer sizes that must be powers of two, or that a table has one entry per enum value.
- `assert(condition)` from `<assert.h>` is checked **at runtime**. If it fails, the program reports the file and line, then stops. Defining `NDEBUG` removes all asserts, which is common for release builds. On a microcontroller you'd usually redirect a failed assert to your own handler that logs the location and resets the device safely.

In [ ]:
%%writefile /content/embedded_c/part3/src/checks.c
#include <stdio.h>
#include <stdint.h>
#include <assert.h>

typedef struct {
    uint8_t  cmd;
    uint8_t  len;
    uint16_t value;
} frame_t;

typedef enum { MODE_IDLE, MODE_RUN, MODE_SLEEP, MODE_COUNT } op_mode_t;
static const char *const mode_names[] = { "IDLE", "RUN", "SLEEP" };

#define BUF_SIZE 8u

/* --- compile-time checks: the build fails if any becomes false --- */
_Static_assert(sizeof(frame_t) == 4, "frame_t must be exactly 4 bytes to match the protocol");
_Static_assert((BUF_SIZE & (BUF_SIZE - 1u)) == 0, "BUF_SIZE must be a power of two");
_Static_assert(sizeof mode_names / sizeof mode_names[0] == MODE_COUNT,
               "mode_names needs one entry per mode");

#ifdef BREAK_BUILD
_Static_assert(sizeof(frame_t) == 3, "demo: this one is deliberately false");
#endif

static uint8_t buffer[BUF_SIZE];

static void write_byte(unsigned index, uint8_t value)
{
    assert(index < BUF_SIZE && "write_byte: index out of range");   /* runtime check */
    buffer[index] = value;
}

int main(void)
{
    printf("all static checks passed at compile time; frame_t = %zu bytes\n", sizeof(frame_t));
    printf("mode %d is %s\n", MODE_SLEEP, mode_names[MODE_SLEEP]);

    write_byte(2, 0xAB);
    printf("buffer[2] = 0x%02X\n", buffer[2]);

    printf("now writing to index 9 of an 8-byte buffer...\n");
    fflush(stdout);
    write_byte(9, 0xCD);                     /* bug: the assert catches it */
    printf("never printed\n");
    return 0;
}

In [ ]:
!cd /content/embedded_c/part3 && echo "=== normal build: the runtime assert fires ===" && gcc -Wall -Wextra -std=c11 src/checks.c -o checks && ./checks 2>&1; echo "(exit code $?)"
!cd /content/embedded_c/part3 && echo "=== -DBREAK_BUILD: a false static assert stops compilation ===" && gcc -Wall -Wextra -std=c11 -DBREAK_BUILD src/checks.c -o checks 2>&1 | grep -E "error|static" | head -n 3

---
## 6. Optimisation levels and code size

Flash is limited, so firmware is usually built with `-Os` (optimise for **size**) rather than `-O2` (for speed). `-O0` (no optimisation) is easiest to step through in a debugger but produces the biggest, slowest code.

| Flag | Typical use |
|---|---|
| `-O0 -g` | debugging: every line maps cleanly to machine code |
| `-Os` | release firmware where Flash is tight (most MCUs) |
| `-O2` | speed-critical code with Flash to spare |
| `-ffunction-sections -fdata-sections` + linker `--gc-sections` | strip unused functions from the final image |

Let's compare the code size of `crc.c` at each level. The table is counted in `bss` because it's built at runtime; making it a `const` array would move it to Flash instead.

In [ ]:
!cd /content/embedded_c/part3 && for opt in O0 O2 Os; do gcc -std=c11 -$opt -c src/crc.c -o crc_$opt.o; done && size crc_O0.o crc_O2.o crc_Os.o

Numbers vary by compiler version and CPU, and they're much smaller for ARM Thumb code. Try the same comparison with the `arm-none-eabi-gcc` toolchain from Part 1, adding `-mcpu=cortex-m0 -mthumb`.

---
# Embedded C rules of thumb

These summarise the three embedded notebooks, and they line up with widely used guidelines like MISRA C and the Barr Group Embedded C Coding Standard:

- Use `stdint.h` types (`uint8_t`, `uint32_t`) whenever the size matters.
- Use `1u << n` for bit masks, and wrap macro arguments in parentheses.
- Every hardware register and every variable shared with an ISR is `volatile`.
- Keep ISRs short: capture data, set a flag or push to a ring buffer, return.
- Protect multi-step updates of shared data with a short critical section.
- Never block in the main loop: use timestamps instead of `delay()`.
- Avoid `malloc` after startup; use static allocation or fixed-block pools.
- Mark tables and strings `const` so they stay in Flash.
- Prefer fixed-point math on chips without an FPU.
- Check data from the outside world: CRCs, range checks, timeouts.
- Let the compiler find bugs: `-Wall -Wextra`, `_Static_assert`, and sanitizers when testing on a PC.
- Separate hardware access (HAL) from logic, so the logic can be tested on a PC, just as these notebooks did.

---
# Exercises

1. **Sections:** move the CRC-16 table into a `const uint16_t crc16_table[256] = {...}` (print the table from the current program and paste it in). Re-run the `size` cell and watch `bss` shrink while `text` grows.
2. **Pool:** add a check to `pool_free` that catches **double frees** (freeing a block that's already free). Hint: keep a bitmap of allocated blocks.
3. **Fixed-point:** write `q16_t q16_sqrt(q16_t x)` using Newton's method with integers only, and compare it with `sqrt()` from `<math.h>`.
4. **Moving average:** build a fixed-point moving-average filter over the last 8 ADC readings using a ring buffer from Part 2. Choose the window size so the division becomes a shift.
5. **Reliability:** write a `config_t` struct that's saved to "Flash" (a file in Colab) along with its CRC-16. On load, reject the data and fall back to defaults if the CRC doesn't match.